# z007 - Drift: variables en pesos como ranking dentro de cada mes

**Idea:** la inflación "corre" los montos de un mes a otro. Pasando cada variable en pesos a su **posición dentro del mes**
(0 = el más bajo, 1 = el más alto), un mismo cliente queda igual aunque los montos suban.

**Comparación** (trial 34 de z005, 5 semillas, train 202104, valid 202106):
- **Actual:** variables originales + lags y deltas.
- **Ranking:** las variables en pesos pasadas a ranking por mes, y lags y deltas calculados sobre esos rankings.

Ambas versiones se corren acá con el mismo código para que la comparación sea justa.

In [1]:
# --- 1. Librerías y parámetros ---
import numpy as np
import pandas as pd
import lightgbm as lgb

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
MESES_TRAIN = [202104]
MES_VALID   = 202106
SEMILLAS    = [501419, 623353, 786349, 834569, 946331]

# Trial 34 de z005 (el mejor en promedio de 5 semillas)
params = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
    "num_leaves": 137, "min_sum_hessian_in_leaf": 9.755574,
}
ARBOLES = 538

In [2]:
# --- 2. Cargar datos y elegir las variables en pesos ---
base = pd.read_parquet("../competencia_01.parquet")
base["clase_binaria"] = (base["clase_ternaria"] == "BAJA+2").astype(int)
base = base.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)

no_usar = ["foto_mes", "clase_ternaria", "clase_binaria"]
cols = [c for c in base.columns if c not in no_usar and c != "numero_de_cliente"]

# Variables en pesos: empiezan con "m" o son montos de tarjetas (Master_m..., Visa_m...)
monetarias = [c for c in cols if c.startswith(("m", "Master_m", "Visa_m"))]
print(len(monetarias), "variables en pesos:", monetarias)

73 variables en pesos: ['mrentabilidad', 'mrentabilidad_annual', 'mcomisiones', 'mactivos_margen', 'mpasivos_margen', 'mcuenta_corriente_adicional', 'mcuenta_corriente', 'mcaja_ahorro', 'mcaja_ahorro_adicional', 'mcaja_ahorro_dolares', 'mcuentas_saldo', 'mautoservicio', 'mtarjeta_visa_consumo', 'mtarjeta_master_consumo', 'mprestamos_personales', 'mprestamos_prendarios', 'mprestamos_hipotecarios', 'mplazo_fijo_dolares', 'mplazo_fijo_pesos', 'minversion1_pesos', 'minversion1_dolares', 'minversion2', 'mpayroll', 'mpayroll2', 'mcuenta_debitos_automaticos', 'mttarjeta_visa_debitos_automaticos', 'mttarjeta_master_debitos_automaticos', 'mpagodeservicios', 'mpagomiscuentas', 'mcajeros_propios_descuentos', 'mtarjeta_visa_descuentos', 'mtarjeta_master_descuentos', 'mcomisiones_mantenimiento', 'mcomisiones_otras', 'mforex_buy', 'mforex_sell', 'mtransferencias_recibidas', 'mtransferencias_emitidas', 'mextraccion_autoservicio', 'mcheques_depositados', 'mcheques_emitidos', 'mcheques_depositados_rech

In [3]:
# --- 3. Función: agregar lags y deltas de orden 1 y 2 ---
def con_lags(df):
    grupo = df.groupby("numero_de_cliente")[cols]
    nuevas = []
    for k in [1, 2]:
        lag = grupo.shift(k).astype("float32")
        delta = (df[cols] - lag).astype("float32")
        lag.columns   = [f"{c}_lag{k}" for c in cols]
        delta.columns = [f"{c}_delta{k}" for c in cols]
        nuevas += [lag, delta]
    return pd.concat([df] + nuevas, axis=1)

def ganancia_max(prob, y):
    ganancia_cliente = np.where(y == 1, GANANCIA_ACIERTO, -COSTO_ESTIMULO)
    acum = np.cumsum(ganancia_cliente[np.argsort(prob)[::-1]])
    n = int(np.argmax(acum)) + 1
    return acum[n - 1] / 1e6, n, acum[11_999] / 1e6

def evaluar(df, nombre):
    variables = [c for c in df.columns if c not in no_usar]
    train = df[df["foto_mes"].isin(MESES_TRAIN)]
    valid = df[df["foto_mes"] == MES_VALID]
    datos = lgb.Dataset(train[variables], label=train["clase_binaria"])
    filas = []
    for semilla in SEMILLAS:
        modelo = lgb.train({**params, "seed": semilla}, datos, num_boost_round=ARBOLES)
        g, n, g12 = ganancia_max(modelo.predict(valid[variables]), valid["clase_binaria"].values)
        filas.append({"version": nombre, "semilla": semilla, "ganancia_max_M": round(g, 1),
                      "mejor_n": n, "top12000_M": round(g12, 1)})
        print(nombre, semilla, round(g, 1))
    return filas

In [4]:
# --- 4. Versión actual (sin ranking) ---
df = con_lags(base.copy())
filas = evaluar(df, "Actual")
del df

Actual 501419 400.5


Actual 623353 397.2


Actual 786349 401.4


Actual 834569 397.2


Actual 946331 401.8


In [5]:
# --- 5. Versión con ranking por mes ---
df = base.copy()
# Para cada mes, posición de cada cliente entre 0 y 1; los NaN siguen siendo NaN
df[monetarias] = df.groupby("foto_mes")[monetarias].rank(pct=True).astype("float32")
df = con_lags(df)   # los lags y deltas se calculan sobre los rankings
filas += evaluar(df, "Ranking")
del df

Ranking 501419 387.3


Ranking 623353 377.4


Ranking 786349 377.4


Ranking 834569 376.4


Ranking 946331 384.6


In [6]:
# --- 6. Comparación ---
comp = pd.DataFrame(filas)
tabla = comp.pivot(index="semilla", columns="version", values="ganancia_max_M")
tabla["diferencia"] = tabla["Ranking"] - tabla["Actual"]
print("Ranking gana en", (tabla["diferencia"] > 0).sum(), "de", len(SEMILLAS), "semillas")
print(comp.groupby("version")[["ganancia_max_M", "top12000_M", "mejor_n"]].agg(["mean", "std"]).round(1))
tabla

Ranking gana en 0 de 5 semillas
        ganancia_max_M      top12000_M       mejor_n        
                  mean  std       mean  std     mean     std
version                                                     
Actual           399.6  2.3      392.9  3.7  13212.0  1072.1
Ranking          380.6  5.0      372.9  4.8  13039.8   957.5


version,Actual,Ranking,diferencia
semilla,,,
501419,400.5,387.3,-13.2
623353,397.2,377.4,-19.8
786349,401.4,377.4,-24.0
834569,397.2,376.4,-20.8
946331,401.8,384.6,-17.2


In [7]:
# --- 7. Anotar en resultados_experimentos.csv ---
ruta_res = "resultados_experimentos.csv"
res = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
res = res[~res["descripcion"].str.startswith("z007")]   # sin duplicar si se vuelve a correr

for version, r in comp.groupby("version").mean(numeric_only=True).iterrows():
    nueva = {
        "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
        "descripcion": f"z007 drift {version}: trial 34, lags 1 y 2 (prom. 5 semillas)",
        "meses_train": str(MESES_TRAIN),
        "mes_valid": MES_VALID,
        "semilla": str(SEMILLAS),
        "params": str(params),
        "mejor_n": int(r["mejor_n"]),
        "ganancia_max_M": round(r["ganancia_max_M"], 1),
    }
    res = pd.concat([res, pd.DataFrame([nueva])], ignore_index=True)

res["n_enviados"] = res["n_enviados"].astype("Int64")
res.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
res.tail(2)

,fecha,descripcion,meses_train,mes_valid,semilla,params,n_corte_0025,ganancia_0025_M,mejor_n,ganancia_max_M,prob_en_corte,auc,ks,lift,captura,n_enviados,ganancia_publica_M,ganancia_top10000_M,ganancia_top12000_M
25,2026-10-07 21:06,"z007 drift Actual: trial 34, lags 1 y 2 (prom....",[202104],202106,"[501419, 623353, 786349, 834569, 946331]","{'objective': 'binary', 'learning_rate': 0.005...",NaN,NaN,13212.0,399.6,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN
26,2026-10-07 21:06,"z007 drift Ranking: trial 34, lags 1 y 2 (prom...",[202104],202106,"[501419, 623353, 786349, 834569, 946331]","{'objective': 'binary', 'learning_rate': 0.005...",NaN,NaN,13039.0,380.6,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN
